# Session 07 – Data Cleaning (Part 2: Outliers & Data Types)

Continuation of Session 06's 400-row Gujarat two-wheeler listings dataset. Random seed: 7.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

bikes = pd.read_csv("../Session-06/bike_listings_s6.csv")
print("Loaded shape:", bikes.shape)
bikes.head()


## Task 1 – Detect outliers with boxplots, IQR, and Z-score

In [ ]:
np.random.seed(7)
bikes_outliers = bikes.copy()
price_idx = np.random.choice(bikes_outliers.index, 10, replace=False)
km_idx = np.random.choice(bikes_outliers.index, 6, replace=False)
bikes_outliers.loc[price_idx, "price_inr"] *= 6
bikes_outliers.loc[km_idx, "km_driven"] = 400000

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bikes_outliers.boxplot(column="price_inr", ax=axes[0])
axes[0].set_title("Price Outliers")
bikes_outliers.boxplot(column="km_driven", ax=axes[1])
axes[1].set_title("Kilometres Driven Outliers")
plt.tight_layout()
plt.show()

for column in ["price_inr", "km_driven"]:
    q1, q3 = bikes_outliers[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    iqr_count = ((bikes_outliers[column] < lower) | (bikes_outliers[column] > upper)).sum()
    z = (bikes_outliers[column] - bikes_outliers[column].mean()) / bikes_outliers[column].std()
    print(column, "| IQR outliers:", iqr_count, "| Z-score > 3:", (z.abs() > 3).sum())


**Methods:** IQR flags values outside Q1 − 1.5×IQR and Q3 + 1.5×IQR. Z-score flags values with absolute score above 3.

## Task 2 – Treat outliers without dropping rows

In [ ]:
bikes_treated = bikes_outliers.copy()
q1, q3 = bikes_treated["price_inr"].quantile([0.25, 0.75])
iqr = q3 - q1
lower_fence, upper_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
bikes_treated["price_capped"] = bikes_treated["price_inr"].clip(lower_fence, upper_fence)

p05, p95 = bikes_treated["price_inr"].quantile([0.05, 0.95])
bikes_treated["price_wins"] = bikes_treated["price_inr"].clip(p05, p95)
bikes_treated["km_log"] = np.log1p(bikes_treated["km_driven"])

comparison = pd.DataFrame({
    "column": ["price_inr", "price_capped", "price_wins", "km_driven", "km_log"],
    "maximum": [bikes_treated[c].max() for c in ["price_inr", "price_capped", "price_wins", "km_driven", "km_log"]],
    "mean": [bikes_treated[c].mean() for c in ["price_inr", "price_capped", "price_wins", "km_driven", "km_log"]]
})
print(comparison)


## Task 3 – Convert data types and compare memory

In [ ]:
bikes_typed = bikes_treated.copy()
bikes_typed["price_inr"] = bikes_typed["price_inr"].round().map(lambda x: f"Rs {int(x):,}")
bikes_typed["price_inr"] = pd.to_numeric(
    bikes_typed["price_inr"].str.replace("Rs", "", regex=False)
    .str.replace(",", "", regex=False).str.strip(),
    errors="coerce"
)
bikes_typed["brand"] = bikes_typed["brand"].astype("category")
bikes_typed["city"] = bikes_typed["city"].astype("category")
bikes_typed["owner_count"] = bikes_typed["owner_count"].astype("int8")
bikes_typed["is_first_owner"] = (bikes_typed["owner_count"] == 1).astype(bool)

print("Dtypes before:")
print(bikes_treated.dtypes)
print("\nDtypes after:")
print(bikes_typed.dtypes)
print("Memory before:", bikes_treated.memory_usage(deep=True).sum(), "bytes")
print("Memory after:", bikes_typed.memory_usage(deep=True).sum(), "bytes")


## Task 4 – Parse mixed date formats and save final CSV

In [ ]:
rng = np.random.default_rng(7)
start_date, end_date = pd.Timestamp("2024-01-01"), pd.Timestamp("2024-12-31")
random_dates = start_date + pd.to_timedelta(
    rng.integers(0, (end_date - start_date).days + 1, size=len(bikes_typed)), unit="D"
)
split_point = int(0.60 * len(bikes_typed))
bikes_typed["listed_on"] = [
    date.strftime("%d/%m/%Y") if i < split_point else date.strftime("%d-%b-%Y")
    for i, date in enumerate(random_dates)
]

date_a = pd.to_datetime(
    bikes_typed["listed_on"].where(bikes_typed.index < split_point),
    format="%d/%m/%Y", errors="coerce"
)
date_b = pd.to_datetime(
    bikes_typed["listed_on"].where(bikes_typed.index >= split_point),
    format="%d-%b-%Y", errors="coerce"
)
bikes_typed["listed_on"] = date_a.fillna(date_b)
bikes_typed["listed_month"] = bikes_typed["listed_on"].dt.month_name()

print("NaT count:", bikes_typed["listed_on"].isna().sum())
assert bikes_typed["listed_on"].isna().sum() == 0
bikes_typed.to_csv("bike_listings_s7.csv", index=False)
print("Saved bike_listings_s7.csv")


## Final checks
- All 400 rows are retained; no rows are dropped.
- `price_inr` is numeric, `brand` and `city` are category, `owner_count` is `int8`, and `is_first_owner` is Boolean.
- Mixed dates are parsed into one datetime column with zero NaT values.
- Final output: `bike_listings_s7.csv`.